In [1]:
import json
import requests
import os
from dotenv import load_dotenv

In [2]:
load_dotenv("../.env")

CHAT_URL = os.getenv("CHAT_URL")
API_KEY = os.getenv("API_KEY")
MODEL = os.getenv("MODEL")

In [3]:
def calculator(expression: str) -> str:
    allowed = set("0123456789+-*/(). ")
    if not set(expression) <= allowed:
        return "Error: expression contains disallowed characters."
    try:
        res = eval(expression, {"__builtins__": {}}, {})
        return str(round(float(res), 2))
    except Exception as e:
        return f"Error: {e}"

**Note**: `round`ing of the float value was done to avoid model calling the `calculator` tool repeatedly 

In [4]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "calculator",
            "description": "Evaluate a basic numerical expression and return the numeric result.",
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {
                        "type": "string",
                        "description": "A basic numerical expression e.g. '42 * 3.1415 / 0.707'",
                    }
                },
                "required": ["expression"],
            },
        },
    }
]
AVAILABLE_FUNCTIONS = {"calculator": calculator}

In [5]:
def call_model(messages, tools=None):
    payload = {"model": MODEL, "messages": messages, "stream": False}
    if tools:
        payload["tools"] = tools

    headers = {"Authorization": f"Bearer {API_KEY}", "Content-Type": "application/json"}

    response = requests.post(CHAT_URL, headers=headers, json=payload, timeout=60)
    print("status code:", response.status_code)
    return response.json()

In [6]:
messages = [
    {
        "role": "system",
        "content": (
            "Use the calculator tool for any arithmetic. "
            "Once you receive the tool result, provide the final answer directly to the user "
            "and do not call the tool again."
        ),
    },
    {"role": "user", "content": "What is 15 percent of 84.50?"},
]
data = call_model(messages=messages, tools=tools)

status code: 200


In [7]:
message = data["choices"][0]["message"]
message

{'role': 'assistant',
 'tool_calls': [{'id': 'call_3155503',
   'type': 'function',
   'function': {'name': 'calculator',
    'arguments': '{"expression":"84.50 * 0.15"}'}}]}

In [8]:
tool_calls = message.get("tool_calls")
if tool_calls:
    messages.append(message)

    for tool_call in tool_calls:
        fn_name = tool_call["function"]["name"]
        fn_args = json.loads(tool_call["function"]["arguments"])

        print(f"Model requested tool call: {fn_name}({fn_args})")

        result = AVAILABLE_FUNCTIONS[fn_name](**fn_args)

        print(f"Tool result: {result}")

        messages.append(
            {
                "role": "tool",
                "tool_call_id": tool_call["id"],
                "content": result,
            }
        )

    final = call_model(messages)
    print(final["choices"][0]["message"]["content"])

else:
    print("Model does not support tool calling.")

Model requested tool call: calculator({'expression': '84.50 * 0.15'})
Tool result: 12.67
status code: 200
15 percent of 84.50 is 12.67.
